# Modelo ensemble Ridge–Gradient Boosting para predicción de NDVI

Este notebook implementa el modelo seleccionado después de comparar diferentes variables objetivo, escenarios de variables, algoritmos y combinaciones.

## Configuración seleccionada

- Variable objetivo: NDVI.
- Escenario: clima más historia de NDVI.
- Número de variables predictoras: 39.
- Modelo 1: Ridge, con `alpha = 100`.
- Modelo 2: Gradient Boosting.
- Peso de Ridge: 55 %.
- Peso de Gradient Boosting: 45 %.
- Validación: cinco particiones temporales crecientes.
- Métrica de selección: RMSE de validación.

El conjunto de prueba final se mantiene reservado y no se utiliza en este notebook.

### 1. importaciones y rutas

In [2]:
import sys
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor

from sklearn.model_selection import (TimeSeriesSplit,GridSearchCV,RandomizedSearchCV,cross_validate)

from sklearn.metrics import (mean_squared_error,mean_absolute_error,r2_score,make_scorer)

# Configuración reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Identificar la raíz del repositorio
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    REPO_ROOT = CURRENT_DIR.parent
else:
    REPO_ROOT = CURRENT_DIR

DATASET_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "dataset_modelo.csv")

MODELS_PATH = REPO_ROOT / "models"
RESULTS_PATH = REPO_ROOT / "results"

# Importar funciones propias del proyecto
sys.path.insert(0, str(MODELS_PATH))

from _experiment_utils import (prepare_features,split_train_val_test)

print("Raíz del repositorio:", REPO_ROOT)
print("Dataset encontrado:", DATASET_PATH.exists())
print("Carpeta models encontrada:", MODELS_PATH.exists())
print("Carpeta results encontrada:", RESULTS_PATH.exists())

Raíz del repositorio: c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe
Dataset encontrado: True
Carpeta models encontrada: True
Carpeta results encontrada: True


### 2. cargar y separar los datos

In [3]:
df = pd.read_csv(DATASET_PATH,parse_dates=["window_start", "window_end"])

train_val, test_final = split_train_val_test(df,fecha_col="window_start",frac_test_final=0.15)

# Ordenar y reconstruir índices para evitar desalineaciones
train_val = (train_val
    .sort_values(["window_start", "region"])
    .reset_index(drop=True))

test_final = (test_final
    .sort_values(["window_start", "region"])
    .reset_index(drop=True))

resumen_datos = pd.DataFrame([{
    "filas_totales": len(df),"filas_train_val": len(train_val),"filas_test_final": len(test_final),
    "inicio_train_val": train_val["window_start"].min(),"fin_train_val": train_val["window_start"].max(),
    "inicio_test_final": test_final["window_start"].min(),"fin_test_final": test_final["window_start"].max(),
    "test_posterior": (train_val["window_start"].max()
        < test_final["window_start"].min())}])

display(resumen_datos)

print("\nObservaciones por región en train_val:")
display(train_val["region"].value_counts().to_frame("observaciones"))

print("\nObservaciones por región en test_final:")
display(test_final["region"].value_counts().to_frame("observaciones"))

assert len(train_val) + len(test_final) == len(df)
assert (train_val["window_start"].max()
    < test_final["window_start"].min())

print("\nSeparación temporal verificada correctamente.")

split_train_val_test (fecha_col='window_start', frac_test_final=0.15):
  train_val:  2000-03-21 00:00:00 -> 2022-07-28 00:00:00  (1030 filas)
  test_final: 2022-08-13 00:00:00 -> 2026-07-12 00:00:00  (182 filas)


,filas_totales,filas_train_val,filas_test_final,inicio_train_val,fin_train_val,inicio_test_final,fin_test_final,test_posterior
0,1212,1030,182,2000-03-21,2022-07-28,2022-08-13,2026-07-12,True



Observaciones por región en train_val:


,observaciones
region,
Cauca,515
Narino,515



Observaciones por región en test_final:


,observaciones
region,
Cauca,91
Narino,91



Separación temporal verificada correctamente.


### 3. preparar las 39 variables del modelo

In [4]:
# Preparar únicamente el conjunto de entrenamiento y validación
X_completo_train, y_ndvi = prepare_features(train_val,target_col="ndvi")

# Variables excluidas para reproducir el escenario seleccionado
columnas_excluir = ["lai_high_lag0","lai_high_lag1","lai_high_lag2","evi_lag_1year","evi_lag1w"]

columnas_excluir = [
    columna
    for columna in columnas_excluir
    if columna in X_completo_train.columns]

X_ndvi = (X_completo_train
    .drop(columns=columnas_excluir)
    .reset_index(drop=True))

y_ndvi = y_ndvi.reset_index(drop=True)

columnas_prohibidas = ["ndvi","evi","window_start","window_end"]

columnas_prohibidas_presentes = [
    columna
    for columna in columnas_prohibidas
    if columna in X_ndvi.columns]

resumen_variables = pd.DataFrame([{"observaciones": len(X_ndvi),"variables": X_ndvi.shape[1],"faltantes_totales": int(X_ndvi.isna().sum().sum()),
    "registros_completos_pct": (
        100 * X_ndvi.notna().all(axis=1).mean()),
    "columnas_prohibidas": columnas_prohibidas_presentes}])

display(resumen_variables)

print("\nVariables excluidas:")
print(columnas_excluir)

print("\nVariables con valores faltantes:")
display(
    X_ndvi.isna()
    .sum()
    .loc[lambda serie: serie > 0]
    .sort_values(ascending=False)
    .to_frame("faltantes"))

assert X_ndvi.shape[1] == 39
assert len(X_ndvi) == len(y_ndvi)
assert len(columnas_prohibidas_presentes) == 0

print("\nEscenario de variables verificado correctamente.")

,observaciones,variables,faltantes_totales,registros_completos_pct,columnas_prohibidas
0,1030,39,88,95.728155,[]



Variables excluidas:
['lai_high_lag0', 'lai_high_lag1', 'lai_high_lag2', 'evi_lag_1year', 'evi_lag1w']

Variables con valores faltantes:


,faltantes
deficit_hidrico_trend2y,44
ndvi_lag_1year,42
et_mm_lag1,2



Escenario de variables verificado correctamente.


### 4. construir los cinco folds temporales

In [5]:
fechas_train_val = (pd.to_datetime(train_val["window_start"])
    .reset_index(drop=True))

fechas_unicas = np.sort(fechas_train_val.unique())

# Cinco bloques de validación y un bloque inicial de entrenamiento
test_size_fechas = len(fechas_unicas) // 6

divisor_temporal = TimeSeriesSplit(n_splits=5,test_size=test_size_fechas)

cv_temporal = []
resumen_folds = []

for numero_fold, (
    idx_fechas_train,
    idx_fechas_val
) in enumerate(
    divisor_temporal.split(fechas_unicas),
    start=1):
    fechas_fold_train = fechas_unicas[idx_fechas_train]
    fechas_fold_val = fechas_unicas[idx_fechas_val]

    idx_train = np.flatnonzero(
        fechas_train_val
        .isin(fechas_fold_train)
        .to_numpy())

    idx_val = np.flatnonzero(
        fechas_train_val
        .isin(fechas_fold_val)
        .to_numpy())

    cv_temporal.append((idx_train, idx_val))

    resumen_folds.append({
        "fold": numero_fold,
        "filas_train": len(idx_train),
        "filas_validacion": len(idx_val),
        "inicio_train": pd.Timestamp(
            fechas_fold_train.min()
        ).date(),
        "fin_train": pd.Timestamp(
            fechas_fold_train.max()
        ).date(),
        "inicio_validacion": pd.Timestamp(
            fechas_fold_val.min()
        ).date(),
        "fin_validacion": pd.Timestamp(
            fechas_fold_val.max()
        ).date(),
        "orden_temporal_correcto": (
            fechas_fold_train.max()
            < fechas_fold_val.min()
        ),
        "indices_compartidos": len(
            set(idx_train).intersection(set(idx_val))
        )})

tabla_folds_temporales = pd.DataFrame(resumen_folds)

display(tabla_folds_temporales)

assert len(cv_temporal) == 5
assert tabla_folds_temporales[
    "orden_temporal_correcto"].all()

assert (tabla_folds_temporales[
        "indices_compartidos"
    ] == 0).all()

print("\nValidación temporal construida correctamente.")

,fold,filas_train,filas_validacion,inicio_train,fin_train,inicio_validacion,fin_validacion,orden_temporal_correcto,indices_compartidos
0,1,180,170,2000-03-21,2004-02-02,2004-02-18,2007-10-16,True,0
1,2,350,170,2000-03-21,2007-10-16,2007-11-01,2011-06-26,True,0
2,3,520,170,2000-03-21,2011-06-26,2011-07-12,2015-03-06,True,0
3,4,690,170,2000-03-21,2015-03-06,2015-03-22,2018-11-17,True,0
4,5,860,170,2000-03-21,2018-11-17,2018-12-03,2022-07-28,True,0



Validación temporal construida correctamente.


### 5. búsqueda de alpha para Ridge con GridSearchCV

In [6]:
pipeline_ridge = Pipeline([
    ("imputador",SimpleImputer(strategy="median")),
    ("escalador",StandardScaler()),
    ("modelo",Ridge())])

# 29 valores entre 0.001 y 10,000
rejilla_ridge = {"modelo__alpha": np.logspace(-3,4,29)}

busqueda_ridge = GridSearchCV(estimator=pipeline_ridge,param_grid=rejilla_ridge,scoring="neg_root_mean_squared_error",
    cv=cv_temporal,refit=True,return_train_score=True,n_jobs=-1)

busqueda_ridge.fit(X_ndvi,y_ndvi)

resultados_ridge = pd.DataFrame(busqueda_ridge.cv_results_)

tabla_ridge = pd.DataFrame({
    "alpha": resultados_ridge[
        "param_modelo__alpha"].astype(float),
    "rmse_train": -resultados_ridge[
        "mean_train_score"],
    "rmse_cv": -resultados_ridge[
        "mean_test_score"],
    "rmse_std": resultados_ridge[
        "std_test_score"],
    "posicion": resultados_ridge[
        "rank_test_score"]})

tabla_ridge["brecha_rmse"] = (tabla_ridge["rmse_cv"]
    - tabla_ridge["rmse_train"])

tabla_ridge = (tabla_ridge
    .sort_values("posicion")
    .reset_index(drop=True))

print("Mejor alpha:",busqueda_ridge.best_params_["modelo__alpha"])
print("Mejor RMSE temporal:",-busqueda_ridge.best_score_)
print("\nDiez mejores configuraciones:")
display(tabla_ridge.head(10).round(6))

Mejor alpha: 100.0
Mejor RMSE temporal: 0.0218831782375258

Diez mejores configuraciones:


,alpha,rmse_train,rmse_cv,rmse_std,posicion,brecha_rmse
0,100.000000,0.019532,0.021883,0.002636,1,0.002352
1,56.234133,0.019210,0.021893,0.002588,2,0.002683
2,177.827941,0.019999,0.021902,0.002678,3,0.001903
3,31.622777,0.018995,0.021921,0.002547,4,0.002926
4,17.782794,0.018856,0.021963,0.002515,5,0.003107
5,316.227766,0.020674,0.021988,0.002697,6,0.001314
6,10.000000,0.018768,0.022010,0.002491,7,0.003241
7,5.623413,0.018712,0.022055,0.002473,8,0.003343
8,3.162278,0.018675,0.022097,0.002459,9,0.003423
9,1.778279,0.018649,0.022136,0.002450,10,0.003487


### 6. calibrar Gradient Boosting con RandomizedSearchCV

In [7]:
pipeline_gradient_boosting = Pipeline([("imputador",SimpleImputer(strategy="median")),
    ("modelo",GradientBoostingRegressor(random_state=RANDOM_STATE))])

distribuciones_gradient_boosting = {
    "modelo__n_estimators": [50, 75, 100, 150, 200, 300],
    "modelo__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.10],
    "modelo__max_depth": [1, 2, 3, 4],
    "modelo__min_samples_leaf": [3, 5, 10, 15, 20],
    "modelo__min_samples_split": [2, 5, 10, 20],
    "modelo__subsample": [0.60, 0.70, 0.80, 0.85, 0.90, 1.00],
    "modelo__max_features": [None, "sqrt", "log2", 0.50, 0.75]}

busqueda_gradient_boosting = RandomizedSearchCV(estimator=pipeline_gradient_boosting,param_distributions=distribuciones_gradient_boosting,
    n_iter=60,scoring="neg_root_mean_squared_error",cv=cv_temporal,refit=True,return_train_score=True,random_state=RANDOM_STATE,n_jobs=-1,verbose=1)

busqueda_gradient_boosting.fit(X_ndvi,y_ndvi)

resultados_gb = pd.DataFrame(busqueda_gradient_boosting.cv_results_)

columnas_resultados_gb = ["param_modelo__n_estimators","param_modelo__learning_rate","param_modelo__max_depth",
    "param_modelo__min_samples_leaf","param_modelo__min_samples_split","param_modelo__subsample",
    "param_modelo__max_features","mean_train_score","mean_test_score","std_test_score","rank_test_score"]

tabla_gb = (resultados_gb[columnas_resultados_gb]
    .copy()
    .rename(columns={
        "param_modelo__n_estimators": "n_estimators","param_modelo__learning_rate": "learning_rate",
        "param_modelo__max_depth": "max_depth","param_modelo__min_samples_leaf": "min_samples_leaf",
        "param_modelo__min_samples_split": "min_samples_split","param_modelo__subsample": "subsample",
        "param_modelo__max_features": "max_features","mean_train_score": "rmse_train",
        "mean_test_score": "rmse_cv","std_test_score": "rmse_std","rank_test_score": "posicion"}))

tabla_gb["rmse_train"] = -tabla_gb["rmse_train"]
tabla_gb["rmse_cv"] = -tabla_gb["rmse_cv"]

tabla_gb["brecha_rmse"] = (
    tabla_gb["rmse_cv"]
    - tabla_gb["rmse_train"])

tabla_gb = (
    tabla_gb
    .sort_values("posicion")
    .reset_index(drop=True))

print("\nMejores hiperparámetros:")
for parametro, valor in (
    busqueda_gradient_boosting
    .best_params_
    .items()):
    print(f"{parametro}: {valor}")

print("\nMejor RMSE temporal:",-busqueda_gradient_boosting.best_score_)
print("\nDiez mejores configuraciones:")
display(tabla_gb.head(10).round(6))

Fitting 5 folds for each of 60 candidates, totalling 300 fits

Mejores hiperparámetros:
modelo__subsample: 0.85
modelo__n_estimators: 50
modelo__min_samples_split: 2
modelo__min_samples_leaf: 15
modelo__max_features: 0.5
modelo__max_depth: 3
modelo__learning_rate: 0.05

Mejor RMSE temporal: 0.02218713051656567

Diez mejores configuraciones:


,n_estimators,learning_rate,max_depth,min_samples_leaf,min_samples_split,subsample,max_features,rmse_train,rmse_cv,rmse_std,posicion,brecha_rmse
0,50,0.05,3,15,2,0.85,0.5,0.016926,0.022187,0.002732,1,0.005261
1,200,0.03,2,20,5,0.90,0.75,0.016074,0.022198,0.003191,2,0.006124
2,200,0.03,2,15,5,0.70,0.75,0.015992,0.022201,0.003143,3,0.006209
3,75,0.03,3,15,5,0.60,sqrt,0.018701,0.022236,0.002319,4,0.003535
4,50,0.08,3,15,5,0.60,0.75,0.015624,0.022242,0.002806,5,0.006618
5,300,0.02,2,10,20,1.00,None,0.015673,0.022285,0.003018,6,0.006612
6,50,0.10,2,15,5,1.00,0.75,0.016420,0.022295,0.002887,7,0.005875
7,100,0.03,4,3,5,0.80,log2,0.013255,0.022377,0.003247,8,0.009122
8,75,0.08,1,20,20,1.00,log2,0.019485,0.022398,0.002437,9,0.002913
9,100,0.03,4,15,10,1.00,0.75,0.013796,0.022436,0.002887,10,0.008640


### 7. comparar Gradient Boosting anterior y calibrado

In [8]:
# Configuración utilizada en el notebook anterior
pipeline_gb_anterior = Pipeline([("imputador",SimpleImputer(strategy="median")),
    ("modelo", GradientBoostingRegressor(n_estimators=100,learning_rate=0.03,
            max_depth=2,min_samples_leaf=10,min_samples_split=2,
            subsample=0.85,max_features=None,random_state=RANDOM_STATE))])

# Mejor configuración encontrada por RandomizedSearchCV
pipeline_gb_randomizado = (busqueda_gradient_boosting.best_estimator_)

modelos_gb_comparar = {"GB configuración anterior": pipeline_gb_anterior,"GB búsqueda aleatoria": pipeline_gb_randomizado}

comparacion_gb = []

for nombre_modelo, modelo in modelos_gb_comparar.items():

    resultados = cross_validate(
        estimator=modelo,
        X=X_ndvi,
        y=y_ndvi,
        cv=cv_temporal,
        scoring={"rmse": "neg_root_mean_squared_error","r2": "r2"},
        return_train_score=True,
        n_jobs=-1)

    comparacion_gb.append({
        "modelo": nombre_modelo,
        "rmse_train": (
            -resultados["train_rmse"].mean()),
        "rmse_cv": (
            -resultados["test_rmse"].mean()),
        "rmse_std": (
            resultados["test_rmse"].std()),
        "r2_cv": resultados["test_r2"].mean(),
        "brecha_rmse": (
            -resultados["test_rmse"].mean()
            + resultados["train_rmse"].mean())})

tabla_comparacion_gb = (
    pd.DataFrame(comparacion_gb)
    .sort_values("rmse_cv")
    .reset_index(drop=True))

display(tabla_comparacion_gb.round(6))

print("\nMejor configuración según RMSE:",tabla_comparacion_gb.loc[0, "modelo"])

,modelo,rmse_train,rmse_cv,rmse_std,r2_cv,brecha_rmse
0,GB configuración anterior,0.017677,0.022033,0.002676,0.574568,0.004356
1,GB búsqueda aleatoria,0.016926,0.022187,0.002732,0.567672,0.005261



Mejor configuración según RMSE: GB configuración anterior


### 8. búsqueda estructural localizada de Gradient Boosting

In [9]:
pipeline_gb_local = Pipeline([("imputador",SimpleImputer(strategy="median")),
    ("modelo",GradientBoostingRegressor(n_estimators=100,learning_rate=0.03,random_state=RANDOM_STATE))])

rejilla_gb_local = {
    "modelo__max_depth": [1, 2, 3],
    "modelo__min_samples_leaf": [5, 10, 15, 20],
    "modelo__min_samples_split": [2, 5],
    "modelo__subsample": [0.75, 0.85, 1.00],
    "modelo__max_features": [None, 0.75]}

busqueda_gb_local = GridSearchCV(estimator=pipeline_gb_local,param_grid=rejilla_gb_local,
    scoring="neg_root_mean_squared_error",cv=cv_temporal,
    refit=True,return_train_score=True,n_jobs=-1,verbose=1)

busqueda_gb_local.fit(X_ndvi,y_ndvi)

resultados_gb_local = pd.DataFrame(busqueda_gb_local.cv_results_)

tabla_gb_local = pd.DataFrame({
    "max_depth": resultados_gb_local[
        "param_modelo__max_depth"],
    "min_samples_leaf": resultados_gb_local[
        "param_modelo__min_samples_leaf"],
    "min_samples_split": resultados_gb_local[
        "param_modelo__min_samples_split"],
    "subsample": resultados_gb_local[
        "param_modelo__subsample"],
    "max_features": resultados_gb_local[
        "param_modelo__max_features"],
    "rmse_train": -resultados_gb_local[
        "mean_train_score"],
    "rmse_cv": -resultados_gb_local[
        "mean_test_score"],
    "rmse_std": resultados_gb_local[
        "std_test_score"],
    "posicion": resultados_gb_local[
        "rank_test_score"]})

tabla_gb_local["brecha_rmse"] = (
    tabla_gb_local["rmse_cv"]
    - tabla_gb_local["rmse_train"])

tabla_gb_local = (tabla_gb_local
    .sort_values("posicion")
    .reset_index(drop=True))

print("\nMejores parámetros estructurales:")

for parametro, valor in (
    busqueda_gb_local
    .best_params_
    .items()):
    print(f"{parametro}: {valor}")

print("\nMejor RMSE temporal:",-busqueda_gb_local.best_score_)
print("\nDiez mejores configuraciones:")
display(tabla_gb_local.head(10).round(6))

Fitting 5 folds for each of 144 candidates, totalling 720 fits

Mejores parámetros estructurales:
modelo__max_depth: 2
modelo__max_features: 0.75
modelo__min_samples_leaf: 15
modelo__min_samples_split: 2
modelo__subsample: 0.75

Mejor RMSE temporal: 0.0218964071462401

Diez mejores configuraciones:


,max_depth,min_samples_leaf,min_samples_split,subsample,max_features,rmse_train,rmse_cv,rmse_std,posicion,brecha_rmse
0,2,15,5,0.75,0.75,0.017940,0.021896,0.002760,1,0.003957
1,2,15,2,0.75,0.75,0.017940,0.021896,0.002760,1,0.003957
2,3,15,2,0.75,0.75,0.016010,0.021916,0.002869,3,0.005906
3,3,15,5,0.75,0.75,0.016010,0.021916,0.002869,3,0.005906
4,2,20,5,0.85,0.75,0.017984,0.021920,0.002667,5,0.003936
5,2,20,2,0.85,0.75,0.017984,0.021920,0.002667,5,0.003936
6,2,20,5,0.85,None,0.017890,0.021928,0.002635,7,0.004037
7,2,20,2,0.85,None,0.017890,0.021928,0.002635,7,0.004037
8,2,15,2,0.85,0.75,0.017813,0.021938,0.002611,9,0.004124
9,2,15,5,0.85,0.75,0.017813,0.021938,0.002611,9,0.004124


### 9. calibrar árboles y tasa de aprendizaje

In [10]:
# Incorporar los mejores parámetros estructurales
pipeline_gb_refinado = clone(pipeline_gb_local)

pipeline_gb_refinado.set_params(**busqueda_gb_local.best_params_)

# Evaluar conjuntamente cantidad de árboles y learning rate
rejilla_gb_refinada = {
    "modelo__n_estimators": [
        50, 75, 100, 125, 150, 200, 250],
    "modelo__learning_rate": [
        0.01, 0.02, 0.03, 0.04,
        0.05, 0.07, 0.10]}

busqueda_gb_refinada = GridSearchCV(
    estimator=pipeline_gb_refinado,
    param_grid=rejilla_gb_refinada,
    scoring="neg_root_mean_squared_error",
    cv=cv_temporal,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=1)

busqueda_gb_refinada.fit(
    X_ndvi,
    y_ndvi)

resultados_gb_refinados = pd.DataFrame(
    busqueda_gb_refinada.cv_results_)

tabla_gb_refinada = pd.DataFrame({
    "n_estimators": resultados_gb_refinados[
        "param_modelo__n_estimators"],
    "learning_rate": resultados_gb_refinados[
        "param_modelo__learning_rate"],
    "rmse_train": -resultados_gb_refinados[
        "mean_train_score"],
    "rmse_cv": -resultados_gb_refinados[
        "mean_test_score"],
    "rmse_std": resultados_gb_refinados[
        "std_test_score"],
    "posicion": resultados_gb_refinados[
        "rank_test_score"]})

tabla_gb_refinada["brecha_rmse"] = (
    tabla_gb_refinada["rmse_cv"]
    - tabla_gb_refinada["rmse_train"])

tabla_gb_refinada = (
    tabla_gb_refinada
    .sort_values("posicion")
    .reset_index(drop=True))

print("\nMejores parámetros refinados:")

for parametro, valor in (
    busqueda_gb_refinada
    .best_params_
    .items()):
    print(f"{parametro}: {valor}")

print(
    "\nMejor RMSE temporal:",
    -busqueda_gb_refinada.best_score_)

print("\nDiez mejores combinaciones:")
display(tabla_gb_refinada
    .head(10)
    .round(6))

Fitting 5 folds for each of 49 candidates, totalling 245 fits

Mejores parámetros refinados:
modelo__learning_rate: 0.03
modelo__n_estimators: 100

Mejor RMSE temporal: 0.0218964071462401

Diez mejores combinaciones:


,n_estimators,learning_rate,rmse_train,rmse_cv,rmse_std,posicion,brecha_rmse
0,100,0.03,0.017940,0.021896,0.002760,1,0.003957
1,150,0.02,0.017907,0.021938,0.002681,2,0.004031
2,125,0.03,0.017267,0.021959,0.002959,3,0.004693
3,75,0.05,0.017273,0.021984,0.002931,4,0.004711
4,125,0.02,0.018600,0.021997,0.002566,5,0.003397
5,75,0.04,0.017959,0.022009,0.002720,6,0.004051
6,200,0.02,0.017055,0.022025,0.002928,7,0.004970
7,50,0.05,0.018613,0.022065,0.002608,8,0.003451
8,100,0.04,0.017083,0.022066,0.002917,9,0.004983
9,150,0.03,0.016729,0.022088,0.003159,10,0.005360


### 10. generar predicciones OOF y buscar los pesos

In [11]:
# Modelos base con hiperparámetros calibrados
modelo_ridge_calibrado = clone(busqueda_ridge.best_estimator_)

modelo_gb_calibrado = clone(busqueda_gb_refinada.best_estimator_)

# Espacios para predicciones fuera de muestra
predicciones_ridge_oof = np.full(len(y_ndvi),np.nan)

predicciones_gb_oof = np.full(len(y_ndvi),np.nan)

fold_oof = np.full(len(y_ndvi),np.nan)

# Entrenar y predecir separadamente en cada fold
for numero_fold, (
    idx_train,
    idx_val
) in enumerate(
    cv_temporal,
    start=1
):
    ridge_fold = clone(modelo_ridge_calibrado)
    gb_fold = clone(modelo_gb_calibrado)
    ridge_fold.fit(X_ndvi.iloc[idx_train],y_ndvi.iloc[idx_train])
    gb_fold.fit(X_ndvi.iloc[idx_train],y_ndvi.iloc[idx_train])
    predicciones_ridge_oof[idx_val] = (ridge_fold.predict(X_ndvi.iloc[idx_val]))
    predicciones_gb_oof[idx_val] = (gb_fold.predict(X_ndvi.iloc[idx_val]))
    fold_oof[idx_val] = numero_fold


def pearson_seguro(y_true, y_pred):
    """Calcula Pearson evitando series constantes."""

    if (
        np.std(y_true) == 0
        or np.std(y_pred) == 0):
        return 0.0

    return float(
        np.corrcoef(y_true, y_pred)[0, 1])

# Evaluar pesos de 0 a 1 en pasos de 0.05
pesos_ridge = np.round(np.arange(0, 1.001, 0.05),2)

resultados_pesos = []

for peso_ridge in pesos_ridge:

    peso_gb = 1 - peso_ridge

    predicciones_ensemble_oof = (
        peso_ridge
        * predicciones_ridge_oof
        + peso_gb
        * predicciones_gb_oof)

    metricas_por_fold = []

    for numero_fold, (
        _,
        idx_val
    ) in enumerate(
        cv_temporal,
        start=1
    ):
        y_fold = y_ndvi.iloc[
            idx_val
        ].to_numpy()

        pred_fold = (
            predicciones_ensemble_oof[
                idx_val
            ])

        rmse_fold = np.sqrt(
            mean_squared_error(y_fold,pred_fold))

        metricas_por_fold.append({
            "rmse": rmse_fold,
            "r2": r2_score(y_fold,pred_fold),
            "pearson": pearson_seguro(y_fold,pred_fold)})

    metricas_por_fold = pd.DataFrame(metricas_por_fold)

    resultados_pesos.append({
        "peso_ridge": peso_ridge,
        "peso_gradient_boosting": peso_gb,
        "rmse_cv": metricas_por_fold[
            "rmse"
        ].mean(),
        "rmse_std": metricas_por_fold[
            "rmse"
        ].std(ddof=0),
        "r2_cv": metricas_por_fold[
            "r2"
        ].mean(),
        "pearson_cv": metricas_por_fold[
            "pearson"
        ].mean()})

tabla_pesos = (
    pd.DataFrame(resultados_pesos)
    .sort_values(
        ["rmse_cv", "r2_cv"],
        ascending=[True, False])
    .reset_index(drop=True))

mejor_peso_ridge = float(
    tabla_pesos.loc[0, "peso_ridge"])

mejor_peso_gb = float(
    tabla_pesos.loc[
        0,
        "peso_gradient_boosting"
    ])

print(
    "Mejor peso Ridge:",
    mejor_peso_ridge)

print(
    "Mejor peso Gradient Boosting:",
    mejor_peso_gb)

print("\nDiez mejores combinaciones:")
display(
    tabla_pesos
    .head(10)
    .round(6))

Mejor peso Ridge: 0.5
Mejor peso Gradient Boosting: 0.5

Diez mejores combinaciones:


,peso_ridge,peso_gradient_boosting,rmse_cv,rmse_std,r2_cv,pearson_cv
0,0.50,0.50,0.021366,0.002719,0.600092,0.791171
1,0.55,0.45,0.021371,0.002712,0.599924,0.791129
2,0.45,0.55,0.021372,0.002726,0.599859,0.791022
3,0.60,0.40,0.021386,0.002704,0.599356,0.790903
4,0.40,0.60,0.021388,0.002733,0.599225,0.790676
5,0.65,0.35,0.021412,0.002695,0.598386,0.790499
6,0.35,0.65,0.021415,0.002739,0.598189,0.790127
7,0.70,0.30,0.021448,0.002687,0.597014,0.789924
8,0.30,0.70,0.021453,0.002744,0.596753,0.789368
9,0.75,0.25,0.021495,0.002678,0.595242,0.789185


### 11. refinamiento de los pesos

In [12]:
pesos_ridge_finos = np.round(np.arange(0.40, 0.601, 0.01),2)

resultados_pesos_finos = []

for peso_ridge in pesos_ridge_finos:

    peso_gb = 1 - peso_ridge

    predicciones_ensemble_oof = (
        peso_ridge
        * predicciones_ridge_oof
        + peso_gb
        * predicciones_gb_oof)

    metricas_por_fold = []

    for numero_fold, (
        _,
        idx_val
    ) in enumerate(
        cv_temporal,
        start=1
    ):
        y_fold = y_ndvi.iloc[
            idx_val
        ].to_numpy()

        pred_fold = (predicciones_ensemble_oof[idx_val])

        metricas_por_fold.append({
            "rmse": np.sqrt(mean_squared_error(y_fold,pred_fold)),
            "r2": r2_score(y_fold,pred_fold),
            "pearson": pearson_seguro(y_fold,pred_fold)})

    metricas_por_fold = pd.DataFrame(metricas_por_fold)

    resultados_pesos_finos.append({
        "peso_ridge": peso_ridge,
        "peso_gradient_boosting": peso_gb,
        "rmse_cv": metricas_por_fold[
            "rmse"
        ].mean(),
        "rmse_std": metricas_por_fold[
            "rmse"
        ].std(ddof=0),
        "r2_cv": metricas_por_fold[
            "r2"
        ].mean(),
        "pearson_cv": metricas_por_fold[
            "pearson"
        ].mean()})

tabla_pesos_finos = (
    pd.DataFrame(
        resultados_pesos_finos)
    .sort_values(
        ["rmse_cv", "r2_cv"],
        ascending=[True, False])
    .reset_index(drop=True))
mejor_peso_ridge = float(tabla_pesos_finos.loc[0,"peso_ridge"])
mejor_peso_gb = float(tabla_pesos_finos.loc[0,"peso_gradient_boosting"])

print("Peso final Ridge:",mejor_peso_ridge)
print("Peso final Gradient Boosting:",mejor_peso_gb)
print("\nDiez mejores pesos refinados:")
display(
    tabla_pesos_finos
    .head(10)
    .round(7))

Peso final Ridge: 0.5
Peso final Gradient Boosting: 0.5

Diez mejores pesos refinados:


,peso_ridge,peso_gradient_boosting,rmse_cv,rmse_std,r2_cv,pearson_cv
0,0.50,0.50,0.021366,0.002719,0.600092,0.791171
1,0.51,0.49,0.021366,0.002718,0.600091,0.791178
2,0.49,0.51,0.021366,0.002720,0.600078,0.791157
3,0.52,0.48,0.021367,0.002716,0.600073,0.791177
4,0.48,0.52,0.021367,0.002722,0.600047,0.791135
5,0.53,0.47,0.021368,0.002715,0.600040,0.791168
6,0.47,0.53,0.021368,0.002723,0.600001,0.791105
7,0.54,0.46,0.021369,0.002713,0.599990,0.791153
8,0.46,0.54,0.021370,0.002725,0.599938,0.791068
9,0.55,0.45,0.021371,0.002712,0.599924,0.791129


### 12. evaluación temporal completa del ensamble calibrado

In [13]:
modelo_ensemble_calibrado = VotingRegressor(
    estimators=[
        (
            "ridge",
            clone(modelo_ridge_calibrado)
        ),
        (
            "gradient_boosting",
            clone(modelo_gb_calibrado)
        )
    ],
    weights=[
        mejor_peso_ridge,
        mejor_peso_gb
    ],
    n_jobs=1
)

metricas_ensemble = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "pearson": make_scorer(
        pearson_seguro
    )
}

resultados_cv_ensemble = cross_validate(
    estimator=modelo_ensemble_calibrado,
    X=X_ndvi,
    y=y_ndvi,
    cv=cv_temporal,
    scoring=metricas_ensemble,
    return_train_score=True,
    n_jobs=-1
)

tabla_ensemble_por_fold = pd.DataFrame({
    "fold": range(1, 6),
    "rmse_train": (
        -resultados_cv_ensemble[
            "train_rmse"
        ]
    ),
    "rmse_validacion": (
        -resultados_cv_ensemble[
            "test_rmse"
        ]
    ),
    "mae_validacion": (
        -resultados_cv_ensemble[
            "test_mae"
        ]
    ),
    "r2_validacion": (
        resultados_cv_ensemble[
            "test_r2"
        ]
    ),
    "pearson_validacion": (
        resultados_cv_ensemble[
            "test_pearson"
        ]
    )
})

rmse_train = (
    -resultados_cv_ensemble[
        "train_rmse"
    ].mean()
)

rmse_cv = (
    -resultados_cv_ensemble[
        "test_rmse"
    ].mean()
)

rmse_std = resultados_cv_ensemble[
    "test_rmse"
].std(ddof=0)

mae_cv = (
    -resultados_cv_ensemble[
        "test_mae"
    ].mean()
)

r2_cv = resultados_cv_ensemble[
    "test_r2"
].mean()

pearson_cv = resultados_cv_ensemble[
    "test_pearson"
].mean()

rmse_pct_media = (
    100 * rmse_cv / y_ndvi.mean()
)

riesgo_base_proxy = (
    1 - pearson_cv**2
)

calidad_datos = (
    X_ndvi.notna()
    .all(axis=1)
    .mean()
)

brecha_rmse = rmse_cv - rmse_train

tabla_resumen_ensemble = pd.DataFrame([{
    "modelo": (
        "Ensamble Ridge + Gradient Boosting calibrado"
    ),
    "peso_ridge": mejor_peso_ridge,
    "peso_gradient_boosting": mejor_peso_gb,
    "rmse_train": rmse_train,
    "rmse_cv": rmse_cv,
    "rmse_std": rmse_std,
    "rmse_pct_media": rmse_pct_media,
    "mae_cv": mae_cv,
    "r2_cv": r2_cv,
    "pearson_cv": pearson_cv,
    "riesgo_base_proxy": riesgo_base_proxy,
    "calidad_datos": calidad_datos,
    "brecha_rmse": brecha_rmse,
    "cumple_r2": r2_cv >= 0.60,
    "cumple_rmse": rmse_pct_media <= 15,
    "cumple_pearson": pearson_cv >= 0.65,
    "cumple_riesgo_base": (
        riesgo_base_proxy <= 0.50
    ),
    "cumple_calidad": calidad_datos >= 0.90
}])

tabla_resumen_ensemble[
    "cumple_todas"
] = (
    tabla_resumen_ensemble[
        "cumple_r2"
    ]
    & tabla_resumen_ensemble[
        "cumple_rmse"
    ]
    & tabla_resumen_ensemble[
        "cumple_pearson"
    ]
    & tabla_resumen_ensemble[
        "cumple_riesgo_base"
    ]
    & tabla_resumen_ensemble[
        "cumple_calidad"
    ]
)

print("Resultados por fold:")
display(
    tabla_ensemble_por_fold.round(6)
)

print("\nResumen del ensamble calibrado:")
display(
    tabla_resumen_ensemble.round(6))

Resultados por fold:


,fold,rmse_train,rmse_validacion,mae_validacion,r2_validacion,pearson_validacion
0,1,0.016799,0.021168,0.016702,0.684026,0.846333
1,2,0.018052,0.020176,0.015686,0.539310,0.742501
2,3,0.018373,0.019792,0.015168,0.714392,0.849370
3,4,0.018519,0.026631,0.019129,0.502518,0.732457
4,5,0.020054,0.019061,0.015061,0.560215,0.785194



Resumen del ensamble calibrado:


,modelo,peso_ridge,peso_gradient_boosting,rmse_train,rmse_cv,rmse_std,rmse_pct_media,mae_cv,r2_cv,pearson_cv,riesgo_base_proxy,calidad_datos,brecha_rmse,cumple_r2,cumple_rmse,cumple_pearson,cumple_riesgo_base,cumple_calidad,cumple_todas
0,Ensamble Ridge + Gradient Boosting calibrado,0.5,0.5,0.01836,0.021366,0.002719,2.833588,0.016349,0.600092,0.791171,0.374048,0.957282,0.003006,True,True,True,True,True,True


### 13. comparación entre ensamble anterior y calibrado

In [14]:
modelo_ensemble_anterior = VotingRegressor(
    estimators=[(
            "ridge",
            Pipeline([
                ("imputador",
                    SimpleImputer(
                        strategy="median"
                    )),
                ("escalador",StandardScaler()),
                ("modelo",Ridge(alpha=100.0))])),
(
            "gradient_boosting",
            clone(pipeline_gb_anterior))
    ],
    weights=[0.55, 0.45],
    n_jobs=1
)

modelos_comparacion = {
    "Ensamble anterior 55/45": (
        modelo_ensemble_anterior
    ),
    "Ensamble calibrado 50/50": (
        modelo_ensemble_calibrado
    )
}

resultados_comparacion = []

for nombre, modelo in (
    modelos_comparacion.items()
):
    resultados = cross_validate(
        estimator=modelo,
        X=X_ndvi,
        y=y_ndvi,
        cv=cv_temporal,
        scoring=metricas_ensemble,
        return_train_score=True,
        n_jobs=-1
    )

    rmse_train_modelo = (
        -resultados["train_rmse"].mean()
    )

    rmse_cv_modelo = (
        -resultados["test_rmse"].mean()
    )

    r2_modelo = resultados[
        "test_r2"
    ].mean()

    pearson_modelo = resultados[
        "test_pearson"
    ].mean()

    rmse_pct_modelo = (
        100
        * rmse_cv_modelo
        / y_ndvi.mean()
    )

    riesgo_modelo = (
        1 - pearson_modelo**2
    )

    resultados_comparacion.append({
        "modelo": nombre,
        "rmse_train": rmse_train_modelo,
        "rmse_cv": rmse_cv_modelo,
        "rmse_pct_media": rmse_pct_modelo,
        "r2_cv": r2_modelo,
        "pearson_cv": pearson_modelo,
        "riesgo_base_proxy": riesgo_modelo,
        "brecha_rmse": (
            rmse_cv_modelo
            - rmse_train_modelo
        ),
        "cumple_todas": (
            (r2_modelo >= 0.60)
            and (rmse_pct_modelo <= 15)
            and (pearson_modelo >= 0.65)
            and (riesgo_modelo <= 0.50)
            and (calidad_datos >= 0.90)
        )
    })

tabla_comparacion_ensembles = (
    pd.DataFrame(
        resultados_comparacion
    )
)

rmse_anterior = tabla_comparacion_ensembles.loc[
    tabla_comparacion_ensembles[
        "modelo"
    ] == "Ensamble anterior 55/45",
    "rmse_cv"
].iloc[0]

rmse_calibrado = tabla_comparacion_ensembles.loc[
    tabla_comparacion_ensembles[
        "modelo"
    ] == "Ensamble calibrado 50/50",
    "rmse_cv"
].iloc[0]

mejora_rmse_porcentual = (
    100
    * (rmse_anterior - rmse_calibrado)
    / rmse_anterior
)

display(
    tabla_comparacion_ensembles.round(6)
)

print(
    "\nReducción porcentual del RMSE:",
    round(mejora_rmse_porcentual, 4),
    "%")

,modelo,rmse_train,rmse_cv,rmse_pct_media,r2_cv,pearson_cv,riesgo_base_proxy,brecha_rmse,cumple_todas
0,Ensamble anterior 55/45,0.018313,0.021417,2.840395,0.598250,0.790540,0.375046,0.003104,False
1,Ensamble calibrado 50/50,0.018360,0.021366,2.833588,0.600092,0.791171,0.374048,0.003006,True



Reducción porcentual del RMSE: 0.2397 %


### 14. evaluación definitiva en test_final

In [15]:
# Preparar el conjunto de prueba final
X_test_completo, y_test = prepare_features(
    test_final,
    target_col="ndvi"
)

X_test = (
    X_test_completo
    .drop(
        columns=[
            columna
            for columna in columnas_excluir
            if columna in X_test_completo.columns
        ]
    )
    .reset_index(drop=True)
)

y_test = y_test.reset_index(drop=True)

# Garantizar el mismo orden de variables
X_test = X_test.reindex(
    columns=X_ndvi.columns
)

assert list(X_test.columns) == list(
    X_ndvi.columns
)

assert X_test.shape[1] == 39
assert len(X_test) == len(test_final)

# Entrenar con todo train_val
modelo_final_calibrado = clone(
    modelo_ensemble_calibrado
)

modelo_final_calibrado.fit(
    X_ndvi,
    y_ndvi
)

# Única predicción sobre test_final
predicciones_test = (
    modelo_final_calibrado.predict(
        X_test
    )
)

rmse_test = np.sqrt(
    mean_squared_error(
        y_test,
        predicciones_test
    )
)

mae_test = mean_absolute_error(
    y_test,
    predicciones_test
)

r2_test = r2_score(
    y_test,
    predicciones_test
)

pearson_test = pearson_seguro(
    y_test,
    predicciones_test
)

rmse_pct_media_test = (
    100 * rmse_test / y_test.mean()
)

riesgo_base_test = (
    1 - pearson_test**2
)

calidad_test = (
    X_test.notna()
    .all(axis=1)
    .mean()
)

tabla_test_final = pd.DataFrame([{
    "periodo_inicio": (
        test_final["window_start"].min()
    ),
    "periodo_fin": (
        test_final["window_start"].max()
    ),
    "observaciones": len(y_test),
    "rmse": rmse_test,
    "rmse_pct_media": rmse_pct_media_test,
    "mae": mae_test,
    "r2": r2_test,
    "pearson": pearson_test,
    "riesgo_base_proxy": riesgo_base_test,
    "calidad_datos": calidad_test,
    "cumple_r2": r2_test >= 0.60,
    "cumple_rmse": (
        rmse_pct_media_test <= 15
    ),
    "cumple_pearson": (
        pearson_test >= 0.65
    ),
    "cumple_riesgo_base": (
        riesgo_base_test <= 0.50
    ),
    "cumple_calidad": (
        calidad_test >= 0.90
    )
}])

tabla_test_final["cumple_todas"] = (
    tabla_test_final["cumple_r2"]
    & tabla_test_final["cumple_rmse"]
    & tabla_test_final["cumple_pearson"]
    & tabla_test_final[
        "cumple_riesgo_base"
    ]
    & tabla_test_final[
        "cumple_calidad"
    ]
)

# Evaluación separada por región
resultados_test_region = []

regiones_test = (
    test_final["region"]
    .reset_index(drop=True)
)

for region in regiones_test.unique():

    mascara = (
        regiones_test == region
    ).to_numpy()

    y_region = y_test[
        mascara
    ]

    pred_region = predicciones_test[
        mascara
    ]

    rmse_region = np.sqrt(
        mean_squared_error(
            y_region,
            pred_region
        )
    )

    pearson_region = pearson_seguro(
        y_region,
        pred_region
    )

    resultados_test_region.append({
        "region": region,
        "observaciones": mascara.sum(),
        "rmse": rmse_region,
        "rmse_pct_media": (
            100
            * rmse_region
            / y_region.mean()
        ),
        "mae": mean_absolute_error(
            y_region,
            pred_region
        ),
        "r2": r2_score(
            y_region,
            pred_region
        ),
        "pearson": pearson_region,
        "riesgo_base_proxy": (
            1 - pearson_region**2
        )
    })

tabla_test_por_region = pd.DataFrame(
    resultados_test_region
)

print("Evaluación general en test_final:")
display(
    tabla_test_final.round(6)
)

print("\nEvaluación en test_final por región:")
display(
    tabla_test_por_region.round(6))

Evaluación general en test_final:


,periodo_inicio,periodo_fin,observaciones,rmse,rmse_pct_media,mae,r2,pearson,riesgo_base_proxy,calidad_datos,cumple_r2,cumple_rmse,cumple_pearson,cumple_riesgo_base,cumple_calidad,cumple_todas
0,2022-08-13,2026-07-12,182,0.023049,2.969551,0.018755,0.281763,0.762042,0.419292,1.0,False,True,True,True,True,False



Evaluación en test_final por región:


,region,observaciones,rmse,rmse_pct_media,mae,r2,pearson,riesgo_base_proxy
0,Cauca,91,0.021189,2.738880,0.017512,0.139772,0.697314,0.513753
1,Narino,91,0.024769,3.180849,0.019998,0.350537,0.801807,0.357106


### 15. guardar el modelo calibrado y sus resultados

In [16]:
# Tabla de hiperparámetros definitivos
parametros_gb_finales = (
    modelo_gb_calibrado
    .named_steps["modelo"]
    .get_params()
)

tabla_hiperparametros_finales = pd.DataFrame([{
    "ridge_alpha": (
        modelo_ridge_calibrado
        .named_steps["modelo"]
        .get_params()["alpha"]
    ),
    "gb_n_estimators": parametros_gb_finales[
        "n_estimators"
    ],
    "gb_learning_rate": parametros_gb_finales[
        "learning_rate"
    ],
    "gb_max_depth": parametros_gb_finales[
        "max_depth"
    ],
    "gb_min_samples_leaf": parametros_gb_finales[
        "min_samples_leaf"
    ],
    "gb_min_samples_split": parametros_gb_finales[
        "min_samples_split"
    ],
    "gb_subsample": parametros_gb_finales[
        "subsample"
    ],
    "gb_max_features": parametros_gb_finales[
        "max_features"
    ],
    "peso_ridge": mejor_peso_ridge,
    "peso_gradient_boosting": mejor_peso_gb
}])

# Predicciones finales para trazabilidad
tabla_predicciones_test = (
    test_final[
        ["window_start", "region"]
    ]
    .reset_index(drop=True)
    .copy()
)

tabla_predicciones_test["ndvi_real"] = (
    y_test.to_numpy()
)

tabla_predicciones_test["ndvi_predicho"] = (
    predicciones_test
)

tabla_predicciones_test["residuo"] = (
    tabla_predicciones_test["ndvi_real"]
    - tabla_predicciones_test["ndvi_predicho"]
)

# Artefacto completo del modelo
artefacto_calibrado = {
    "modelo": modelo_final_calibrado,
    "objetivo": "ndvi",
    "escenario": "clima_mas_historia_ndvi",
    "variables": X_ndvi.columns.tolist(),
    "numero_variables": X_ndvi.shape[1],
    "hiperparametros": (
        tabla_hiperparametros_finales
        .iloc[0]
        .to_dict()
    ),
    "metodos_calibracion": {
        "ridge": "GridSearchCV",
        "gradient_boosting_inicial": (
            "RandomizedSearchCV"
        ),
        "gradient_boosting_refinamiento": (
            "GridSearchCV"
        ),
        "pesos_ensemble": (
            "busqueda exhaustiva mediante "
            "predicciones OOF"
        )
    },
    "metricas_cv": (
        tabla_resumen_ensemble
        .iloc[0]
        .to_dict()
    ),
    "metricas_test_final": (
        tabla_test_final
        .iloc[0]
        .to_dict()
    ),
    "periodo_train_val": {
        "inicio": str(
            train_val["window_start"].min()
        ),
        "fin": str(
            train_val["window_start"].max()
        )
    },
    "periodo_test_final": {
        "inicio": str(
            test_final["window_start"].min()
        ),
        "fin": str(
            test_final["window_start"].max()
        )
    }
}

MODELS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

ruta_modelo = (
    MODELS_PATH
    / "ensemble_ridge_gradient_boosting_ndvi_calibrado.joblib"
)

ruta_hiperparametros = (
    RESULTS_PATH
    / "hiperparametros_ensemble_ndvi_calibrado.csv"
)

ruta_metricas_cv = (
    RESULTS_PATH
    / "metricas_ensemble_ndvi_calibrado_cv.csv"
)

ruta_metricas_folds = (
    RESULTS_PATH
    / "metricas_ensemble_ndvi_calibrado_por_fold.csv"
)

ruta_metricas_test = (
    RESULTS_PATH
    / "metricas_ensemble_ndvi_calibrado_test.csv"
)

ruta_metricas_test_region = (
    RESULTS_PATH
    / "metricas_ensemble_ndvi_calibrado_test_region.csv"
)

ruta_predicciones_test = (
    RESULTS_PATH
    / "predicciones_ensemble_ndvi_calibrado_test.csv"
)

joblib.dump(
    artefacto_calibrado,
    ruta_modelo
)

tabla_hiperparametros_finales.to_csv(
    ruta_hiperparametros,
    index=False
)

tabla_resumen_ensemble.to_csv(
    ruta_metricas_cv,
    index=False
)

tabla_ensemble_por_fold.to_csv(
    ruta_metricas_folds,
    index=False
)

tabla_test_final.to_csv(
    ruta_metricas_test,
    index=False
)

tabla_test_por_region.to_csv(
    ruta_metricas_test_region,
    index=False
)

tabla_predicciones_test.to_csv(
    ruta_predicciones_test,
    index=False
)

print("Archivos guardados correctamente:\n")

for ruta in [
    ruta_modelo,
    ruta_hiperparametros,
    ruta_metricas_cv,
    ruta_metricas_folds,
    ruta_metricas_test,
    ruta_metricas_test_region,
    ruta_predicciones_test
]:
    print(ruta)

Archivos guardados correctamente:

c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\models\ensemble_ridge_gradient_boosting_ndvi_calibrado.joblib
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\results\hiperparametros_ensemble_ndvi_calibrado.csv
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\results\metricas_ensemble_ndvi_calibrado_cv.csv
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\results\metricas_ensemble_ndvi_calibrado_por_fold.csv
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\results\metricas_ensemble_ndvi_calibrado_test.csv
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\res

## Conclusión de la calibración y evaluación final

La calibración se realizó exclusivamente con el conjunto `train_val` y cinco particiones temporales crecientes. Para Ridge se aplicó `GridSearchCV` sobre 29 valores de `alpha`, seleccionándose `alpha=100`. Para Gradient Boosting se efectuó primero una exploración amplia con `RandomizedSearchCV`, evaluando 60 configuraciones. Posteriormente, se realizaron dos búsquedas localizadas con `GridSearchCV`: una sobre 144 combinaciones de parámetros estructurales y otra sobre 49 combinaciones del número de árboles y la tasa de aprendizaje. La configuración seleccionada utilizó 100 árboles, tasa de aprendizaje de 0.03, profundidad máxima de 2, mínimo de 15 observaciones por hoja, `min_samples_split=2`, `subsample=0.75` y `max_features=0.75`.

Finalmente, se calibraron los pesos mediante predicciones fuera de muestra de los cinco folds. La combinación seleccionada asignó un 50% a Ridge y un 50% a Gradient Boosting. Durante la validación temporal, el ensamble calibrado obtuvo un RMSE de 0.02137, equivalente al 2.83% de la media del NDVI, un R² de 0.6001 y una correlación de Pearson de 0.7912. Estos resultados representaron una mejora pequeña frente al ensamble anterior y permitieron cumplir las cinco metas definidas durante la validación.

La evaluación definitiva se realizó una única vez sobre `test_final`, correspondiente al periodo entre agosto de 2022 y julio de 2026. En este conjunto, el modelo obtuvo un RMSE de 0.02305, equivalente al 2.97% de la media, una correlación de Pearson de 0.7620, un riesgo base proxy de 0.4193 y una calidad de datos del 100%. Sin embargo, el R² disminuyó a 0.2818, por lo que el modelo cumplió cuatro de las cinco metas, pero no alcanzó el R² mínimo de 0.60 en datos futuros. El desempeño fue mejor en Nariño que en Cauca, lo cual evidencia limitaciones de generalización temporal y regional. En consecuencia, el modelo conserva utilidad predictiva, pero no debe afirmarse que satisface completamente los criterios finales y no se utilizará `test_final` para realizar nuevos ajustes.
